In [ ]:
"""
R004_IMLS_OADS__
|
R004_arxiv_extraction_200_sample.ipynb
Created on Tue Oct 14 23:26:39 2025
@author: Lemos
"""

# URL Extraction Across Document Formats

The 200-paper benchmark was constructed using a stratified sampling approach designed to ensure balanced temporal coverage, diverse document layouts, and representation of edge-case formats. Sampling was conducted in two phases: Set A (2016—2024) and Set B (1992—2015), with each phase contributing 100 papers. In both phases, candidate papers were first stratified by publication year and document layout (single-column, double-column, ETD, and, for older papers, scanned PDFs), and only papers containing at least three embedded URLs were considered. Document layouts were automatically classified using the `pdf_layout_classifier_and_sampler.py` script (included in this repository under `scripts/benchmark_200/`), which analyzes each PDF's page geometry, column boundaries, and text block positions to assign an initial layout category. The automatically assigned labels were then manually verified to correct misclassifications and confirm the final document type. The automatically assigned labels were then manually reviewed to correct misclassifications, identify mixed-layout and other edge cases, and confirm the final document type. Candidate papers were further filtered to retain only those with available LaTeX source files and successful conversion to all required formats (TEXT, HTML, XML, Markdown, and LaTeX). From the verified pool, the final 200-paper benchmark was randomly selected while preserving the desired distribution across layout categories and time periods, resulting in 85 single-column papers, 86 double-column papers, 19 ETDs, and 10 scanned documents spanning 1992--2024. The resulting benchmark contains 2,420 embedded URLs (2,338 unique).

## What this notebook does

For each of the document formats below, the notebook:

1. **Env setup** — installs only the packages needed for that format
2. **Conversion** — turns the source (PDF/LaTeX into the target format (where applicable)
3. **Extraction** — extracts URLs from that format

Formats covered (in order):

| # | Format | Tool | Section |
|---|---|---|---|
| 1 | Text (plain) | PyMuPDF  | [Text](#Text) |
| 2 | Text + annotation layer | PyMuPDF  | [TEXTWAL — PyMuPDF](#TEXTWAL---PyMuPDF) |
| 3 | Text + annotation layer | PyPDF | [TEXTWAL — PyPDF](#TEXTWAL---PyPDF) |
| 4 | Text + annotation layer | pdfminer.six | [TEXTWAL — pdfminer.six](#TEXTWAL---pdfminer.six) |
| 5 | Text + annotation layer, LLM-assisted | Claude (Anthropic API) | [TEXTWAL-CL](#TEXTWAL-CL---Claude-assisted-extraction) |
| 6 | LaTeX source | LaTeXML + pylatexenc + regex | [LaTeX](#LaTeX) |
| 7 | HTML | BeautifulSoup | [HTML](#HTML) |
| 8 | XML (TEI) | GROBID + lxml | [XML (GROBID)](#XML-(GROBID)) |
| 9 | PNG - VLM-assisted | Qwen2-VL-7B-Instruct | [Qwen2-VL](#PNG---Qwen2-VL-7B-Instruct) |
| 10 | PNG - VLM-assisted | DeepSeek-VL-7B-Chat | [DeepSeek-VL](#PNG---DeepSeek-VL-7B-Chat) |
| 11 | PNG - VLM-assisted | MiniCPM-o-2_6 | [MiniCPM](#PNG---MiniCPM-o-2_6) |
| 12 | Markdown | Marker | [Markdown (Marker)](#Markdown-(Marker)) |

The notebook finishes by merging every format's results into a single per-paper JSON file and a union CSV.



## Data layout

This notebook assumes (and will create, where noted) the following structure, **relative to the repository root**. The benchmark dataset containing the 200 sampled arXiv papers (PDFs and corresponding LaTeX source files) can be downloaded from:

https://huggingface.co/datasets/dblind-data/arxiv-url-bench-hf/blob/main/arxiv-200-benchmark/arxiv-url-bench-200-raw-files.tar.gz

After extracting the archive, place the contents so that the PDF files and their corresponding LaTeX source directories are located under `data/200_sample/raw_files/` as shown below

```
data/
└── 200_sample/
    ├── arxiv_extracted_urls_all_formats_200.json   # FINAL combined output
    │
    ├── intermediate_results/                        # one JSON per format
    │   ├── stage_text_urls.json
    │   ├── stage_textwal_pymupdf_urls.json
    │   ├── stage_textwal_pypdf_urls.json
    │   ├── stage_textwal_pdfminer_urls.json
    │   ├── stage_textwalcl_urls.json
    │   ├── stage_html_urls.json
    │   ├── stage_latex_urls.json
    │   ├── stage_xml_grobid_urls.json
    │   ├── stage_vlm_qwen_urls.json
    │   ├── stage_vlm_deepseek_urls.json
    │   ├── stage_vlm_minicpm_urls.json
    │   └── stage_markdown_urls.json
    │
    └── raw_files/
        ├── pdf/             # REQUIRED input: 200 benchmark PDFs (<arxiv_id>.pdf)
        ├── html/            # generated
        ├── latex/           # REQUIRED input: corresponding LaTeX source, one subfolder per paper
        ├── text/            # generated
        ├── textwal/         # generated (pymupdf / pypdf / pdfminer subfolders)
        ├── xml/             # generated (GROBID TEI output)
        ├── markdown/        # generated (Marker output)
        └── vlm_png/         # generated (page images shared by all VLMs)
```

Only `data/200_sample/raw_files/pdf/` and `data/200_sample/raw_files/latex/` need to be populated before running this notebook. These directories should contain the 200-paper benchmark downloaded from the Hugging Face repository. All remaining directories and intermediate outputs are created automatically during notebook execution.

## Environments

Several formats need **conflicting** package versions and cannot share a
single Python environment. Separate `requirements/*.txt` files are provided
for each; create a dedicated virtual environment / conda environment per
row before running that section:

| Environment | Requirements file | Used by |
|---|---|---|
| Core | `requirements/requirements_core.txt` | Text, TEXTWAL (all variants), LaTeX, HTML, XML, final merge |
| VLM — Qwen | `requirements/requirements_vlm_qwen.txt` | Qwen2-VL section |
| VLM — DeepSeek | `requirements/requirements_vlm_deepseek.txt` | DeepSeek-VL section |
| VLM — MiniCPM | `requirements/requirements_vlm_minicpm.txt` | MiniCPM-o section |
| Marker | `requirements/requirements_marker.txt` | Markdown section |

For the Claude-assisted extraction (TEXTWAL-CL) section, create a `.env`
file at the repository root containing:

```
ANTHROPIC_API_KEY=your-key-here
```

## Global setup

Paths, the shared URL-matching regex, and small helpers used throughout the
rest of the notebook. Run this section first, regardless of which format
section(s) you run afterwards.

In [2]:
import json
import re
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Folder layout (relative to the repository root — see overview above)
# ---------------------------------------------------------------------------
DATA_DIR   = Path("data/200_sample")
RAW_DIR    = DATA_DIR / "raw_files"
INTER_DIR  = DATA_DIR / "intermediate_results"
FINAL_JSON = DATA_DIR / "arxiv_extracted_urls_all_formats_200.json"

PDF_DIR              = RAW_DIR / "pdf"                    # REQUIRED: 200 source PDFs, <arxiv_id>.pdf
TEXT_DIR             = RAW_DIR / "text"
TEXTWAL_PYMUPDF_DIR  = RAW_DIR / "textwal" / "pymupdf"
TEXTWAL_PYPDF_DIR    = RAW_DIR / "textwal" / "pypdf"
TEXTWAL_PDFMINER_DIR = RAW_DIR / "textwal" / "pdfminer"
HTML_DIR             = RAW_DIR / "html"                   # generated by LaTeXML
LATEX_DIR            = RAW_DIR / "latex"                  # REQUIRED: one subfolder per paper
XML_DIR              = RAW_DIR / "xml"                    # generated by GROBID
MARKDOWN_DIR         = RAW_DIR / "markdown"               # generated by Marker
VLM_PNG_DIR          = RAW_DIR / "vlm_png"                
VLM_QWEN_DIR         = RAW_DIR / "vlm" / "qwen"
VLM_DEEPSEEK_DIR     = RAW_DIR / "vlm" / "deepseek"
VLM_MINICPM_DIR      = RAW_DIR / "vlm" / "minicpm"

for _d in [PDF_DIR, TEXT_DIR, TEXTWAL_PYMUPDF_DIR, TEXTWAL_PYPDF_DIR, TEXTWAL_PDFMINER_DIR,
           HTML_DIR, LATEX_DIR, XML_DIR, MARKDOWN_DIR, VLM_PNG_DIR,
           VLM_QWEN_DIR, VLM_DEEPSEEK_DIR, VLM_MINICPM_DIR, INTER_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# The 200 arXiv IDs this notebook operates on, derived from the PDFs present
# ---------------------------------------------------------------------------
def get_arxiv_ids(pdf_dir: Path = PDF_DIR) -> list:
    """Sorted list of arXiv IDs, one per PDF found in `pdf_dir`."""
    ids = sorted(p.stem for p in pdf_dir.glob("*.pdf"))
    print(f"Found {len(ids)} PDFs in {pdf_dir}")
    return ids

ARXIV_IDS = get_arxiv_ids()

Found 200 PDFs in data/200_sample/raw_files/pdf


In [3]:
# ---------------------------------------------------------------------------
# Shared URL-matching regex, used (with small per-format additions) across
# every text-based format: text, textwal, latex, markdown.
# ---------------------------------------------------------------------------
URL_PATTERN = re.compile(r"""(?xi)
    \b(?:                                           # start of URL boundary
        (?:https?|ftp|file|data|javascript|mailto|tel|git|ssh|magnet)://  # protocols
        | www\d{0,3}[.]                             # www. without protocol
        | [a-z0-9.\-]+[.][a-z]{2,4}/                # domain without protocol
    )
    (?:\S+(?::\S*)?@)?                              # user:pass authentication
    (?:
        (?!(?:10|127)(?:\.\d{1,3}){3})               # exclude private/local networks
        (?!(?:169\.254|192\.168)(?:\.\d{1,3}){2})
        (?!172\.(?:1[6-9]|2\d|3[0-1])(?:\.\d{1,3}){2})
        (?:[1-9]\d?|1\d\d|2[01]\d|22[0-3])          # IP address
        (?:\.(?:1?\d{1,2}|2[0-4]\d|25[0-5])){2}
        (?:\.(?:[1-9]\d?|1\d\d|2[0-4]\d|25[0-4]))
    |
        (?:                                         # hostname
            (?:
                [a-z0-9\u00a1-\uffff]               # unicode domain support
                [a-z0-9\u00a1-\uffff_-]{0,62}
            )?
            [a-z0-9\u00a1-\uffff]\.
        )*
        (?:[a-z\u00a1-\uffff]{2,}\.?)               # domain name
    )
    (?::\d{2,5})?                                   # port number
    (?:[/?#][^\s]*)?                                # resource path
    \b                                              # end of URL boundary
    """)

def find_urls(text: str) -> list:
    """Apply URL_PATTERN to `text` and return a de-duplicated list of matches."""
    if not text:
        return []
    return list(set(URL_PATTERN.findall(text)))

# ---------------------------------------------------------------------------
# JSON / progress helpers
# ---------------------------------------------------------------------------
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print(f"Saved -> {path}")

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def summarize(per_paper: dict, label: str):
    """Quick sanity-check stats: how many papers / URLs this format found."""
    total_urls = sum(v["url_count"] for v in per_paper.values())
    papers_with_urls = sum(1 for v in per_paper.values() if v["url_count"] > 0)
    print(f"[{label}] papers processed: {len(per_paper)} | "
          f"papers with >=1 URL: {papers_with_urls} | total URLs: {total_urls}")

def progress(i, n, every=25):
    if i % every == 0 or i == n:
        print(f"  [{i}/{n}]")

## Text

Baseline plain-text extraction using PyMuPDF's `get_text()` — no annotation
layers, metadata, or hidden content (see the TEXTWAL sections below for those).

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Conversion

In [6]:
import pymupdf  # PyMuPDF

def convert_pdf_to_text(pdf_path: Path, text_path: Path):
    doc = pymupdf.open(pdf_path)
    content = "\n".join(page.get_text() for page in doc)
    doc.close()
    text_path.write_text(content, encoding="utf-8")

def convert_all_to_text(pdf_dir: Path = PDF_DIR, out_dir: Path = TEXT_DIR, arxiv_ids=ARXIV_IDS):
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        convert_pdf_to_text(pdf_dir / f"{arxiv_id}.pdf", out_dir / f"{arxiv_id}.txt")
        progress(i, len(arxiv_ids))

convert_all_to_text()

  [25/200]
  [50/200]
  [75/200]
  [100/200]
  [125/200]
  [150/200]
  [175/200]
  [200/200]


### Extraction

In [7]:
def extract_urls_text(text_dir: Path = TEXT_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    """Regex-based URL extraction over all 200 plain-text files at once."""
    results = {}
    for arxiv_id in arxiv_ids:
        txt_path = text_dir / f"{arxiv_id}.txt"
        content = txt_path.read_text(encoding="utf-8") if txt_path.exists() else ""
        urls = find_urls(content)
        results[arxiv_id] = {"filename": str(txt_path), "url_count": len(urls), "urls": urls}
    return results

text_urls = extract_urls_text()
summarize(text_urls, "text")
save_json(text_urls, INTER_DIR / "stage_text_urls.json")

[text] papers processed: 200 | papers with >=1 URL: 190 | total URLs: 1043
Saved -> data/200_sample/intermediate_results/stage_text_urls.json


## TEXTWAL - PyMuPDF

"TEXTWAL" = **Text With Annotation Layer**. This variant goes beyond plain
text and also pulls in PDF annotations, optional-content-group (hidden)
layers, document metadata, and a raw byte-level URL scan of the PDF itself —
several arXiv papers embed clickable links as annotations rather than as
visible text, so this step recovers URLs the baseline `Text` pass misses.

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Conversion

In [10]:
import pymupdf

URL_REGEX_BYTES = re.compile(rb"(https?://[^\s<>()\"']+)", re.IGNORECASE)

def _extract_annotations_pymupdf(page):
    text = ""
    annots = page.annots()
    if not annots:
        return ""
    for annot in annots:
        try:
            text += "\n--- Annotation ---\n"
            raw = annot.parent.parent.xref_object(annot.xref)
            text += raw + "\n"
        except Exception:
            pass
    return text

def _extract_optional_layers_pymupdf(doc):
    text = ""
    try:
        ocgs = doc.get_ocgs()
        if not ocgs:
            return ""
        for name in ocgs.get("ocgs", {}):
            try:
                text += f"\n--- OCG Layer: {name} ---\n"
                layer_text = doc.get_ocg_content(name)
                if layer_text:
                    text += layer_text + "\n"
            except Exception:
                pass
    except Exception:
        pass
    return text

def convert_pdf_textwal_pymupdf(pdf_path: Path, out_path: Path):
    doc = pymupdf.open(pdf_path)
    output = f"=== Extracted from {pdf_path} ===\n"

    for page_num, page in enumerate(doc):
        output += f"\n\n=== PAGE {page_num + 1} TEXT ===\n"
        output += page.get_text("text") + "\n"
        output += _extract_annotations_pymupdf(page)

    output += "\n\n=== OPTIONAL CONTENT LAYERS ===\n"
    output += _extract_optional_layers_pymupdf(doc)

    metadata = doc.metadata or {}
    output += "\n\n=== METADATA ===\n"
    for k, v in metadata.items():
        if v:
            output += f"{k}: {v}\n"

    # Raw byte scan fallback: catches URLs embedded in objects not walked above
    urls = set(u.decode("utf-8", errors="ignore")
               for u in URL_REGEX_BYTES.findall(pdf_path.read_bytes()))
    output += "\n\n=== RAW BYTE SCAN URLS ===\n" + "\n".join(sorted(urls))

    doc.close()
    out_path.write_text(output, encoding="utf-8")

def convert_all_textwal_pymupdf(pdf_dir: Path = PDF_DIR, out_dir: Path = TEXTWAL_PYMUPDF_DIR,
                                 arxiv_ids=ARXIV_IDS):
    out_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        convert_pdf_textwal_pymupdf(pdf_dir / f"{arxiv_id}.pdf", out_dir / f"{arxiv_id}.txt")
        progress(i, len(arxiv_ids))
    print(f"Done in {time.perf_counter() - t0:.1f}s "
          f"({(time.perf_counter() - t0) / len(arxiv_ids):.2f}s/PDF)")

convert_all_textwal_pymupdf()

### Extraction

In [9]:
def extract_urls_textwal_pymupdf(text_dir: Path = TEXTWAL_PYMUPDF_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    results = {}
    for arxiv_id in arxiv_ids:
        txt_path = text_dir / f"{arxiv_id}.txt"
        content = txt_path.read_text(encoding="utf-8") if txt_path.exists() else ""
        urls = find_urls(content)
        results[arxiv_id] = {"filename": str(txt_path), "url_count": len(urls), "urls": urls}
    return results

textwal_pymupdf_urls = extract_urls_textwal_pymupdf()
summarize(textwal_pymupdf_urls, "textwal_pymupdf")
save_json(textwal_pymupdf_urls, INTER_DIR / "stage_textwal_pymupdf_urls.json")

[textwal_pymupdf] papers processed: 200 | papers with >=1 URL: 193 | total URLs: 3187
Saved -> data/200_sample/intermediate_results/stage_textwal_pymupdf_urls.json


## TEXTWAL - PyPDF

Same idea as the PyMuPDF variant above (page text + annotations + metadata +
raw byte scan), implemented with the `pypdf` library instead, as a second,
independent tool for comparison.

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Conversion

In [16]:
from pypdf import PdfReader

def _extract_annotations_pypdf(page):
    text = ""
    if "/Annots" not in page:
        return ""
    for annot_ref in page["/Annots"]:
        try:
            annot = annot_ref.get_object()
            text += "\n--- Annotation ---\n"
            for k, v in annot.items():
                text += f"{k}: {v}\n"
        except Exception:
            pass
    return text

def convert_pdf_textwal_pypdf(pdf_path: Path, out_path: Path):
    reader = PdfReader(str(pdf_path))
    output = f"=== Extracted from {pdf_path} ===\n"

    for page_num, page in enumerate(reader.pages):
        output += f"\n\n=== PAGE {page_num + 1} TEXT ===\n"
        try:
            output += (page.extract_text() or "") + "\n"
        except Exception:
            pass
        output += _extract_annotations_pypdf(page)

    output += "\n\n=== METADATA ===\n"
    meta = reader.metadata or {}
    for k, v in meta.items():
        output += f"{k}: {v}\n"

    urls = set(u.decode("utf-8", errors="ignore")
               for u in URL_REGEX_BYTES.findall(pdf_path.read_bytes()))
    output += "\n\n=== RAW BYTE SCAN URLS ===\n" + "\n".join(sorted(urls))

    out_path.write_text(output, encoding="utf-8")

def convert_all_textwal_pypdf(pdf_dir: Path = PDF_DIR, out_dir: Path = TEXTWAL_PYPDF_DIR,
                               arxiv_ids=ARXIV_IDS):
    out_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        convert_pdf_textwal_pypdf(pdf_dir / f"{arxiv_id}.pdf", out_dir / f"{arxiv_id}.txt")
        progress(i, len(arxiv_ids))
    print(f"Done in {time.perf_counter() - t0:.1f}s")

convert_all_textwal_pypdf()

### Extraction

In [ ]:
def extract_urls_textwal_pypdf(text_dir: Path = TEXTWAL_PYPDF_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    results = {}
    for arxiv_id in arxiv_ids:
        txt_path = text_dir / f"{arxiv_id}.txt"
        content = txt_path.read_text(encoding="utf-8") if txt_path.exists() else ""
        urls = find_urls(content)
        results[arxiv_id] = {"filename": str(txt_path), "url_count": len(urls), "urls": urls}
    return results

textwal_pypdf_urls = extract_urls_textwal_pypdf()
summarize(textwal_pypdf_urls, "textwal_pypdf")
save_json(textwal_pypdf_urls, INTER_DIR / "stage_textwal_pypdf_urls.json")

## TEXTWAL - pdfminer.six

Third independent tool for the same "text + annotation layer" extraction
task: full-document text (best-in-class via `pdfminer.six`), document info
dictionary, a page-object scan, and the same raw byte-level fallback.

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Conversion

In [ ]:
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdfpage import PDFPage

def _extract_metadata_pdfminer(pdf_path: Path):
    output = ""
    try:
        with open(pdf_path, "rb") as f:
            doc = PDFDocument(PDFParser(f))
            for info in doc.info or []:
                for k, v in info.items():
                    output += f"{k}: {v}\n"
    except Exception:
        pass
    return output

def _extract_page_objects_pdfminer(pdf_path: Path):
    output = ""
    try:
        with open(pdf_path, "rb") as f:
            doc = PDFDocument(PDFParser(f))
            for page_num, page in enumerate(PDFPage.create_pages(doc)):
                try:
                    output += f"\n--- PAGE {page_num + 1} OBJECT ---\n{repr(page)}\n"
                except Exception:
                    pass
    except Exception:
        pass
    return output

def convert_pdf_textwal_pdfminer(pdf_path: Path, out_path: Path):
    output = f"=== Extracted from {pdf_path} ===\n"

    try:
        output += "\n\n=== FULL DOCUMENT TEXT ===\n" + (extract_text(str(pdf_path)) or "")
    except Exception as e:
        output += f"\n[Text extraction failed: {e}]\n"

    output += "\n\n=== METADATA ===\n" + _extract_metadata_pdfminer(pdf_path)
    output += "\n\n=== PAGE OBJECT SCAN ===\n" + _extract_page_objects_pdfminer(pdf_path)

    urls = set(u.decode("utf-8", errors="ignore")
               for u in URL_REGEX_BYTES.findall(pdf_path.read_bytes()))
    output += "\n\n=== RAW BYTE SCAN URLS ===\n" + "\n".join(sorted(urls))

    out_path.write_text(output, encoding="utf-8")

def convert_all_textwal_pdfminer(pdf_dir: Path = PDF_DIR, out_dir: Path = TEXTWAL_PDFMINER_DIR,
                                  arxiv_ids=ARXIV_IDS):
    out_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        convert_pdf_textwal_pdfminer(pdf_dir / f"{arxiv_id}.pdf", out_dir / f"{arxiv_id}.txt")
        progress(i, len(arxiv_ids))
    print(f"Done in {time.perf_counter() - t0:.1f}s")

convert_all_textwal_pdfminer()

### Extraction

In [ ]:
def extract_urls_textwal_pdfminer(text_dir: Path = TEXTWAL_PDFMINER_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    results = {}
    for arxiv_id in arxiv_ids:
        txt_path = text_dir / f"{arxiv_id}.txt"
        content = txt_path.read_text(encoding="utf-8") if txt_path.exists() else ""
        urls = find_urls(content)
        results[arxiv_id] = {"filename": str(txt_path), "url_count": len(urls), "urls": urls}
    return results

textwal_pdfminer_urls = extract_urls_textwal_pdfminer()
summarize(textwal_pdfminer_urls, "textwal_pdfminer")
save_json(textwal_pdfminer_urls, INTER_DIR / "stage_textwal_pdfminer_urls.json")

## TEXTWAL-CL - Claude assisted extraction

Instead of a regex, this variant sends the comprehensive PyMuPDF extraction
(text + annotations + hidden layers + metadata, reusing the "Conversion"
step from the PyMuPDF TEXTWAL section above) to Claude and asks it to
extract and reconstruct every URL and DOI it finds. This tends to catch
URLs that were split across lines or reconstructed via hyphenation, which
pure regexes miss.

Requires an `ANTHROPIC_API_KEY` in a `.env` file at the repository root
(see the [Environments](#Environments) note in the overview).

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Conversion

In [ ]:
# Reuses `convert_pdf_textwal_pymupdf` (comprehensive PyMuPDF extraction)
# from the "TEXTWAL — PyMuPDF" section above -- rerun that section's
# Conversion cell first if you have not already, or simply point
# `TEXTWAL_PYMUPDF_DIR` at existing output.
print(f"Using comprehensive-content files from: {TEXTWAL_PYMUPDF_DIR}")
print(f"Files available: {len(list(TEXTWAL_PYMUPDF_DIR.glob('*.txt')))}")

### Extraction

In [ ]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Approximate Claude Haiku pricing (USD per 1M tokens) -- update as needed
INPUT_COST_PER_MILLION = 1.00
OUTPUT_COST_PER_MILLION = 5.00

URL_EXTRACTION_PROMPT = """You are an expert data extraction assistant. Your goal is to analyze the provided academic paper content and extract every unique URL and DOI link found within the text.

### Extraction Rules
1. **Scope**: Extract URLs from the main body, footnotes, references, metadata, and tables.
2. **Reconstruction**: If a URL is split across two lines or contains a hyphen due to text wrapping (e.g., `https://example.com/sub-` on one line and `directory` on the next), reconstruct it into a single, continuous string.
3. **DOIs**: Include Digital Object Identifiers. If a DOI is provided as a raw string (e.g., `10.1038/s41586-020-2012-7`), format it as a full URL: `https://doi.org/[DOI]`.
4. **Validation**: Ensure the links are complete. Remove trailing punctuation (like periods or closing parentheses) that are not part of the actual web address.
5. **De-duplication**: Provide only unique URLs.

### Output Format
Return ONLY a valid JSON array of strings. Do not include introductory text, explanations, or Markdown code blocks.

Format: ["https://url1.com", "https://url2.org"]

### Content to Analyze
{content}
"""

def calculate_cost(input_tokens, output_tokens):
    return round((input_tokens / 1_000_000) * INPUT_COST_PER_MILLION
                 + (output_tokens / 1_000_000) * OUTPUT_COST_PER_MILLION, 6)

def extract_urls_with_claude(content: str, filename: str):
    """Send one paper's comprehensive extracted content to Claude and parse
    the JSON URL array it returns."""
    try:
        prompt = URL_EXTRACTION_PROMPT.format(content=content[:600_000])
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=8192,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )
        response_text = response.content[0].text.strip()
        if response_text.startswith("```"):
            response_text = response_text.split("```")[1]
            if response_text.startswith("json"):
                response_text = response_text[4:]
            response_text = response_text.strip()

        urls = json.loads(response_text)
        if not isinstance(urls, list):
            urls = []
        return urls, response.usage.input_tokens, response.usage.output_tokens, None
    except json.JSONDecodeError as e:
        return [], 0, 0, f"JSON parse error for {filename}: {e}"
    except Exception as e:
        return [], 0, 0, f"Claude API error for {filename}: {e}"

def extract_urls_textwalcl(text_dir: Path = TEXTWAL_PYMUPDF_DIR, arxiv_ids=ARXIV_IDS,
                            sleep_between_calls: float = 1.0) -> dict:
    results = {}
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        txt_path = text_dir / f"{arxiv_id}.txt"
        content = txt_path.read_text(encoding="utf-8") if txt_path.exists() else ""

        t0 = time.time()
        urls, in_tok, out_tok, error = extract_urls_with_claude(content, arxiv_id)
        runtime = round(time.time() - t0, 2)

        results[arxiv_id] = {
            "filename": str(txt_path),
            "url_count": len(urls),
            "urls": urls,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "cost_usd": calculate_cost(in_tok, out_tok),
            "runtime_sec": runtime,
            "failed": error is not None,
        }
        if error:
            print(f"  [{i}/{len(arxiv_ids)}] {arxiv_id}: {error}")
        progress(i, len(arxiv_ids))
        time.sleep(sleep_between_calls)
    return results

textwalcl_urls = extract_urls_textwalcl()
summarize(textwalcl_urls, "textwalcl")
print(f"Total cost: ${sum(v['cost_usd'] for v in textwalcl_urls.values()):.4f}")
save_json(textwalcl_urls, INTER_DIR / "stage_textwalcl_urls.json")

## LaTeX

Uses arXiv's original LaTeX source (`.tex` and `.bbl` files), grouped by
paper — a single paper's source often spans several `.tex`/`.bbl` files
(main file, macros, bibliography, sub-sections), so these are grouped by
arXiv ID before URL extraction.

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Mapping

LaTeX source is sourced directly from arXiv. This step groups every `.tex`/`.bbl` file under `raw_files/latex/` by arXiv ID, so multi-file sources are read together during extraction.

In [19]:
def group_latex_files(latex_dir: Path = LATEX_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    """Map arxiv_id -> list of .tex/.bbl file paths belonging to that paper."""
    ids = set(arxiv_ids)
    file_dict = {}
    for path in latex_dir.rglob("*"):
        if path.suffix not in (".tex", ".bbl"):
            continue
        # First path component under latex_dir is the arxiv_id subfolder
        try:
            arxiv_id = path.relative_to(latex_dir).parts[0]
        except (ValueError, IndexError):
            continue
        if arxiv_id in ids:
            file_dict.setdefault(arxiv_id, []).append(path)

    print(f"LaTeX source found for {len(file_dict)}/{len(arxiv_ids)} papers")
    return file_dict

LATEX_FILE_GROUPS = group_latex_files()

LaTeX source found for 200/200 papers


### Extraction

In [23]:
from pylatexenc.latex2text import LatexNodes2Text

def extract_urls_from_tex_bbl(file_groups: dict) -> dict:
    """For each paper: pull \\url{}/\\urladdr{} targets directly out of the
    raw LaTeX, plus any bare URLs found after converting to plain text."""
    extracted = {}
    for arxiv_id, paths in file_groups.items():
        urls = set()
        for file_path in paths:
            try:
                content = file_path.read_text(encoding="utf-8", errors="ignore")
            except Exception as e:
                print(f"  error reading {file_path}: {e}")
                continue

            plain_text = content
            if file_path.suffix == ".tex":
                try:
                    plain_text = LatexNodes2Text().latex_to_text(content)
                except Exception:
                    pass

            urls.update(re.findall(r"\\url\{([^}]+)\}", content))
            urls.update(re.findall(r"\\urladdr\{([^}]+)\}", content))
            urls.update(find_urls(plain_text))

        extracted[arxiv_id] = list(set(urls))
    return extracted

def extract_urls_latex(file_groups: dict = None, arxiv_ids=ARXIV_IDS) -> dict:
    file_groups = file_groups or LATEX_FILE_GROUPS
    raw = extract_urls_from_tex_bbl(file_groups)
    results = {}
    for arxiv_id in arxiv_ids:
        urls = raw.get(arxiv_id, [])
        results[arxiv_id] = {"url_count": len(urls), "urls": urls}
    return results

latex_urls = extract_urls_latex()
summarize(latex_urls, "latex")
save_json(latex_urls, INTER_DIR / "stage_latex_urls.json")

## HTML

The HTML representation is generated from the LaTeX sources using **LaTeXML**. Each paper's HTML lives in its own subfolder under `data/200_sample/raw_files/html/<arxiv_id>/`. 

### Env setup

Before running any conversion, pull the LaTeXML docker image.

On HPC systems (using Apptainer):

```bash id="eh4wkr"
apptainer pull latexml.sif docker://latexml/ar5ivist:latest
```

Alternatively, in Docker-enabled environments:

```bash id="92odwz"
docker pull latexml/ar5ivist:latest
```

Check **_https://github.com/brucemiller/latexml_** for alternative LaTeXML installation approaches.

### Conversion

Run the following SLURM job:

```bash id="axks8n"
sbatch scripts/benchmark_200/convert_latex_to_html.sh
```

This job executes:

```text id="9o3b2t"
scripts/benchmark_200/convert_latex_to_html.py
```

which converts the LaTeX source of each arXiv paper into HTML using the LaTeXML container.

In [ ]:
def find_html_files(html_dir: Path = HTML_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    """Map arxiv_id -> path of its HTML file (one subfolder per paper)."""
    html_files = {}
    missing = []
    for arxiv_id in arxiv_ids:
        folder = html_dir / arxiv_id
        matches = list(folder.glob("*.html")) if folder.is_dir() else []
        if matches:
            html_files[arxiv_id] = matches[0]
        else:
            missing.append(arxiv_id)
    print(f"HTML found for {len(html_files)}/{len(arxiv_ids)} papers")
    if missing:
        print(f"  missing for: {missing[:10]}{' ...' if len(missing) > 10 else ''}")
    return html_files

HTML_FILES = find_html_files()

### Extraction

In [ ]:
from bs4 import BeautifulSoup

def extract_urls_from_html(html_path: Path, arxiv_id: str) -> list:
    """Pull every link out of the LaTeXML `ltx_page_content` div"""
    try:
        html_content = html_path.read_text(encoding="utf-8")
    except Exception:
        return []

    soup = BeautifulSoup(html_content, "html.parser")
    page_content = soup.find("div", class_="ltx_page_content")
    if not page_content:
        return []

    urls = []
    for a_tag in page_content.find_all("a", href=True):
        url = a_tag["href"]
        if re.search(rf"(arxiv\.org/(abs|pdf|html)/{re.escape(arxiv_id)})", url) or "#" in url:
            continue
        urls.append(url)
    return list(set(urls))

def extract_urls_html(html_files: dict = None, arxiv_ids=ARXIV_IDS) -> dict:
    html_files = html_files or HTML_FILES
    results = {}
    for arxiv_id in arxiv_ids:
        html_path = html_files.get(arxiv_id)
        urls = extract_urls_from_html(html_path, arxiv_id) if html_path else []
        results[arxiv_id] = {
            "filename": str(html_path) if html_path else None,
            "url_count": len(urls),
            "urls": urls,
        }
    return results

html_urls = extract_urls_html()
summarize(html_urls, "html")
save_json(html_urls, INTER_DIR / "stage_html_urls.json")

## XML (GROBID)

Converts each PDF to TEI-XML using [GROBID](https://github.com/kermitt2/grobid) and extracts every `target="..."` attribute from the resulting TEI tree.

### Env setup

In [ ]:
%pip install -q -r requirements/requirements_core.txt

### Conversion

In [ ]:
import requests

# Public demo endpoint.
GROBID_URL = "https://kermitt2-grobid.hf.space/api/processFulltextDocument"

def convert_pdf_to_xml_grobid(pdf_path: Path, xml_path: Path, max_retries: int = 3):
    if xml_path.exists():
        return "skipped"

    for attempt in range(max_retries):
        try:
            with open(pdf_path, "rb") as pdf_file:
                response = requests.post(
                    GROBID_URL,
                    files={"input": pdf_file},
                    params={"includeRawCitations": "true"},
                    timeout=180,
                )
            if response.status_code == 200:
                xml_path.write_text(response.text, encoding="utf-8")
                return "ok"
        except requests.exceptions.RequestException:
            time.sleep(5 * (attempt + 1))  # exponential backoff

    # All retries failed: write a placeholder so re-runs skip it (mark as failed downstream)
    xml_path.write_text(f"<!-- Failed to process {pdf_path.name} -->\n", encoding="utf-8")
    return "failed"

def convert_all_to_xml_grobid(pdf_dir: Path = PDF_DIR, out_dir: Path = XML_DIR, arxiv_ids=ARXIV_IDS):
    out_dir.mkdir(parents=True, exist_ok=True)
    status_counts = {"ok": 0, "skipped": 0, "failed": 0}
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        status = convert_pdf_to_xml_grobid(pdf_dir / f"{arxiv_id}.pdf", out_dir / f"{arxiv_id}.xml")
        status_counts[status] += 1
        if i % 10 == 0:
            time.sleep(3)  # be polite to the shared demo endpoint
        progress(i, len(arxiv_ids))
    print("GROBID conversion:", status_counts)

convert_all_to_xml_grobid()

### Extraction

In [ ]:
from lxml import etree

TEI_NSMAP = {"tei": "http://www.tei-c.org/ns/1.0"}

def extract_urls_from_tei(xml_path: Path) -> list:
    if not xml_path.exists() or xml_path.stat().st_size == 0:
        return []
    try:
        root = etree.parse(str(xml_path)).getroot()
    except etree.XMLSyntaxError:
        return []

    urls = []
    for elem in root.xpath("//*[@target]", namespaces=TEI_NSMAP):
        url = elem.get("target")
        if url and "tei" not in url.lower() and not url.startswith("#"):
            urls.append(url)
    return list(set(urls))

def extract_urls_xml(xml_dir: Path = XML_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    results = {}
    for arxiv_id in arxiv_ids:
        xml_path = xml_dir / f"{arxiv_id}.xml"
        urls = extract_urls_from_tei(xml_path)
        results[arxiv_id] = {"filename": str(xml_path), "url_count": len(urls), "urls": urls}
    return results

xml_urls = extract_urls_xml()
summarize(xml_urls, "xml")
save_json(xml_urls, INTER_DIR / "stage_xml_grobid_urls.json")

## VLM — Vision-Language Models

Three vision-language models are used to extract URLs directly from rendered page **images**: Qwen2-VL, DeepSeek-VL, and MiniCPM-o. Each needs its own, mutually-incompatible Python environment (see [Environments](#Environments) above), so each has its own **Env setup** and **Extraction** subsection below — but the PDF → PNG conversion is identical and only needs to run once for all three.

### Conversion (shared by all three VLMs)

Renders every page of every PDF to a PNG at 2x zoom, saved under `raw_files/vlm_png/<arxiv_id>/page_NNN.png`. This only needs to be run once under any environment that has `pymupdf` installed.

In [ ]:
import pymupdf

def convert_pdf_to_pngs(pdf_path: Path, out_dir: Path, zoom: float = 2.0):
    out_dir.mkdir(parents=True, exist_ok=True)
    doc = pymupdf.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
        pix.save(out_dir / f"page_{page_num + 1:03d}.png")
    n_pages = len(doc)
    doc.close()
    return n_pages

def convert_all_to_pngs(pdf_dir: Path = PDF_DIR, out_dir: Path = VLM_PNG_DIR, arxiv_ids=ARXIV_IDS):
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        n_pages = convert_pdf_to_pngs(pdf_dir / f"{arxiv_id}.pdf", out_dir / arxiv_id)
        if i % 25 == 0 or i == len(arxiv_ids):
            print(f"  [{i}/{len(arxiv_ids)}] {arxiv_id}: {n_pages} pages")

def get_page_images(arxiv_id: str, png_dir: Path = VLM_PNG_DIR) -> list:
    return sorted((png_dir / arxiv_id).glob("page_*.png"))

convert_all_to_pngs()

## PNG - Qwen2-VL-7B-Instruct

### Qwen2-VL-7B-Instruct — Env setup

In [ ]:
# Run this cell in the "VLM - Qwen" environment
# (requirements/requirements_vlm_qwen.txt)
%pip install -q -r requirements/requirements_vlm_qwen.txt

#### Download the model

In [ ]:
# Download Qwen2-VL locally
from huggingface_hub import snapshot_download

print("Downloading Qwen2-VL...")
snapshot_download(
    repo_id="Qwen/Qwen2-VL-7B-Instruct", 
    local_dir="VLMs/qwen2_vl",
    local_dir_use_symlinks=False,
    ignore_patterns=["*.safetensors.index.json", "*.md"]
)
print("Download complete.")

### Extraction

Run the following SLURM job:

```bash
sbatch scripts/benchmark_200/arxiv_vlm_url_extractor_qwen.sh
```

This job executes:

```
scripts/benchmark_200/arxiv_vlm_url_extractor_qwen.py
```

which iterates through the pre-converted PNG directories for the 200 benchmark papers and extracts URLs. This writes one JSON per paper to `data/200_sample/raw_files/vlm_qwen/<arxiv_id>.json`, and writes the aggregated result to `data/200_sample/intermediate_results/stage_vlm_qwen_urls.json`

In [32]:
vlm_qwen_urls = json.load(open(INTER_DIR / "stage_vlm_qwen_urls.json"))
summarize(vlm_qwen_urls, "vlm_qwen")

[vlm_qwen] papers processed: 200 | papers with >=1 URL: 197 | total URLs: 7379


## PNG - DeepSeek-VL-7B-Chat

### DeepSeek-VL-7B-Chat — Env setup

In [ ]:
# Run this cell in the "VLM - DeepSeek" environment
# (requirements/requirements_vlm_deepseek.txt). DeepSeek-VL ships its own
# package that must be installed from source, into a path RELATIVE to the
# repo root:
#
#   git clone https://github.com/deepseek-ai/DeepSeek-VL external/DeepSeek-VL
#   pip install -r requirements/requirements_vlm_deepseek.txt
#   pip install -e external/DeepSeek-VL
%pip install -q -r requirements/requirements_vlm_deepseek.txt
import subprocess
from pathlib import Path as _Path
if not _Path("VLMs/DeepSeek-VL").exists():
    subprocess.run(["git", "clone", "https://github.com/deepseek-ai/DeepSeek-VL",
                     "VLMs/DeepSeek-VL"], check=True)
subprocess.run(["pip", "install", "-e", "VLMs/DeepSeek-VL"], check=True)

In [ ]:
%cd VLMs/DeepSeek-VL

#### Download the model

In [ ]:
# Download deepseek-vl-7b locally
from transformers import AutoModelForCausalLM
from deepseek_vl.models import VLChatProcessor

model_name = "deepseek-ai/deepseek-vl-7b-chat"
local_dir = "VLMs/deepseek-vl-7b-chat"

VLChatProcessor.from_pretrained(model_name).save_pretrained(local_dir)
AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).save_pretrained(local_dir)

print("Model saved to:", local_dir)

### Extraction

Run the following SLURM job:

```bash
sbatch scripts/benchmark_200/arxiv_vlm_url_extractor_deepseek.sh
```

This job executes:

```
scripts/benchmark_200/arxiv_vlm_url_extractor_deepseek.py
```

which iterates through the pre-converted PNG directories for the 200 benchmark papers and extracts URLs. This writes one JSON per paper to `data/200_sample/raw_files/vlm_deepseek/<arxiv_id>.json`, and writes the aggregated result to `data/200_sample/intermediate_results/stage_vlm_deepseek_urls.json`

In [6]:
vlm_deepseek_urls = json.load(open(INTER_DIR / "stage_vlm_deepseek_urls.json"))
summarize(vlm_deepseek_urls, "vlm_deepseek")

[vlm_deepseek] papers processed: 200 | papers with >=1 URL: 194 | total URLs: 8108


## PNG - MiniCPM-o-2_6

### MiniCPM-o-2_6 — Env setup

In [ ]:
# Run this cell in the "VLM - MiniCPM" environment
# (requirements/requirements_vlm_minicpm.txt)
%pip install -q -r requirements/requirements_vlm_minicpm.txt

#### Download the model

In [ ]:
# Download MiniCPM-o-2_6 locally
from huggingface_hub import snapshot_download

LOCAL_MODEL_DIR = "VLMs/MiniCPM-o-2_6"

# Download model once
if not os.path.exists(LOCAL_MODEL_DIR):
    print("Downloading MiniCPM-O model locally...")
    snapshot_download(
        repo_id="openbmb/MiniCPM-o-2_6",
        local_dir=LOCAL_MODEL_DIR,
        local_dir_use_symlinks=False,
        allow_patterns=["*"]
    )
    print("Model saved to:", LOCAL_MODEL_DIR)
else:
    print("Local model already exists:", LOCAL_MODEL_DIR)


### Extraction

Run the following SLURM job:

```bash
sbatch scripts/benchmark_200/arxiv_vlm_url_extractor_minicpm.sh
```

This job executes:

```
scripts/benchmark_200/arxiv_vlm_url_extractor_minicpm.py
```

which iterates through the pre-converted PNG directories for the 200 benchmark papers and extracts URLs. This writes one JSON per paper to `data/200_sample/raw_files/vlm_minicpm/<arxiv_id>.json`, and writes the aggregated result to `data/200_sample/intermediate_results/stage_vlm_minicpm_urls.json`

In [9]:
vlm_minicpm_urls = json.load(open(INTER_DIR / "stage_vlm_minicpm_urls.json"))
summarize(vlm_minicpm_urls, "vlm_minicpm")

[vlm_minicpm] papers processed: 200 | papers with >=1 URL: 199 | total URLs: 29263


## Markdown (Marker)

Converts each PDF to Markdown using [Marker](https://github.com/datalab-to/marker),
then extracts URLs from four complementary patterns: Markdown link/image
syntax, embedded HTML attributes, bare URLs, and the same comprehensive
regex used by the text-based formats above.

### Env setup

In [ ]:
# Run this cell in the "marker" environment (requirements/requirements_marker.txt)
%pip install -q -r requirements/requirements_marker.txt

### Conversion

Run the following SLURM job:

```bash
sbatch scripts/benchmark_200/convert_pdf_to_markdown.sh
```

This job executes:

```
scripts/benchmark_200/convert_pdf_to_markdown.py
```

which converts the PDF of each arXiv paper into Markdown using the Marker library.

### Extraction

In [11]:
MD_LINK_RE = re.compile(r'!?\[(?:[^\[\]]*)\]\((https?://[^\s)]+)\)', re.IGNORECASE)
HTML_ATTR_RE = re.compile(
    r'(?:href|src|action|data-href|data-src)\s*=\s*["\']?(https?://[^\s"\'<>)]+)["\']?', re.IGNORECASE
)
BARE_URL_RE = re.compile(
    r"(?<![(\[\"'])https?://(?:[a-zA-Z0-9\-._~:/?#\[\]@!$&'()*+,;=%]+)", re.IGNORECASE
)

def extract_urls_from_markdown(md_text: str) -> list:
    raw = set()
    raw.update(MD_LINK_RE.findall(md_text))
    raw.update(HTML_ATTR_RE.findall(md_text))
    raw.update(BARE_URL_RE.findall(md_text))
    raw.update(find_urls(md_text))

    cleaned = set()
    for url in raw:
        url = re.sub(r"[.,;:!?\)\]\}'\"]+$", "", url)
        if url:
            cleaned.add(url)
    return sorted(cleaned)

def extract_urls_markdown(md_dir: Path = MARKDOWN_DIR, arxiv_ids=ARXIV_IDS) -> dict:
    results = {}
    for arxiv_id in arxiv_ids:
        md_path = md_dir / f"{arxiv_id}.md"
        text = md_path.read_text(encoding="utf-8") if md_path.exists() else ""
        urls = extract_urls_from_markdown(text)
        results[arxiv_id] = {
            # "filename": f"{arxiv_id}.pdf",
            "filename": str(md_path),
            "url_count": len(urls),
            "urls": urls,
        }
    return results

markdown_urls = extract_urls_markdown()
summarize(markdown_urls, "markdown")
save_json(markdown_urls, INTER_DIR / "stage_markdown_urls.json")

[markdown] papers processed: 200 | papers with >=1 URL: 193 | total URLs: 2159
Saved -> data/200_sample/intermediate_results/stage_markdown_urls.json


## Combine all formats → final 200-paper dataset

Merges every per-format stage JSON in `intermediate_results/` into a single
per-paper record, matching the layout used by `la_360k_sample`'s combined
output at a 200-paper scale. Run this after running (or loading previously
saved results from) every format section above.

In [12]:
STAGE_FILES = {
    "text":             INTER_DIR / "stage_text_urls.json",
    "textwal_pymupdf":  INTER_DIR / "stage_textwal_pymupdf_urls.json",
    "textwal_pypdf":    INTER_DIR / "stage_textwal_pypdf_urls.json",
    "textwal_pdfminer": INTER_DIR / "stage_textwal_pdfminer_urls.json",
    "textwalcl":        INTER_DIR / "stage_textwalcl_urls.json",
    "html":             INTER_DIR / "stage_html_urls.json",
    "latex":            INTER_DIR / "stage_latex_urls.json",
    "xml":              INTER_DIR / "stage_xml_grobid_urls.json",
    "vlm_qwen":         INTER_DIR / "stage_vlm_qwen_urls.json",
    "vlm_deepseek":     INTER_DIR / "stage_vlm_deepseek_urls.json",
    "vlm_minicpm":      INTER_DIR / "stage_vlm_minicpm_urls.json",
    "markdown":         INTER_DIR / "stage_markdown_urls.json",
}

EMPTY_RECORD = {"url_count": 0, "urls": []}

def combine_all_formats(arxiv_ids=ARXIV_IDS, stage_files: dict = None) -> dict:
    stage_files = stage_files or STAGE_FILES
    stage_data = {}
    for format_key, path in stage_files.items():
        if path.exists():
            stage_data[format_key] = load_json(path)
        else:
            print(f"  (missing, will fill with empty records: {path})")
            stage_data[format_key] = {}

    combined = {}
    for arxiv_id in arxiv_ids:
        combined[arxiv_id] = {
            format_key: per_paper.get(arxiv_id, EMPTY_RECORD)
            for format_key, per_paper in stage_data.items()
        }
    return combined

combined_200 = combine_all_formats()
save_json(combined_200, FINAL_JSON)
print(f"\nTotal papers in final dataset: {len(combined_200)}")

Saved -> data/200_sample/arxiv_extracted_urls_all_formats_200.json

Total papers in final dataset: 200


In [24]:
# Per-format summary
for format_key in STAGE_FILES:
    total = sum(len(v[format_key]["urls"]) for v in combined_200.values())
    print(f"{format_key:<18} total URL candidates: {total:>6}")